In [4]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Connexion DuckDB
con = duckdb.connect(
    "../data/processed/chess.duckdb"
)

# Parties
games = con.sql("""
SELECT *
FROM games_matched
""").df()

# Année (utile plus tard)
games["year"] = games["date"].dt.year

print(f"Games: {len(games):,}")

# Vérifier les colonnes liées aux ouvertures
cols = [
    c for c in games.columns
    if "eco" in c.lower()
    or "opening" in c.lower()
]

print("\nOpening columns:")
print(cols)

games[
    ["eco"]
].head()

Games: 483,962

Opening columns:
['eco']


,eco
0,C70
1,C55
2,B33
3,C22
4,D20


In [6]:
opening_stats = []

for eco, group in games.groupby("eco"):

    games_count = len(group)

    if games_count < 500:
        continue

    white_wins = (
        group["result"] == "1-0"
    ).sum()

    draws = (
        group["result"] == "1/2-1/2"
    ).sum()

    white_score = (
        white_wins +
        0.5 * draws
    ) / games_count

    opening_stats.append([
        eco,
        games_count,
        white_score
    ])

opening_stats = pd.DataFrame(
    opening_stats,
    columns=[
        "eco",
        "games",
        "white_score"
    ]
)

opening_stats = opening_stats.sort_values(
    "white_score",
    ascending=False
)

opening_stats.head(20)

,eco,games,white_score
173,E17,710,0.617606
24,A35,832,0.611779
162,D78,1094,0.603291
145,D11,1902,0.600158
147,D15,1066,0.598030
69,B31,2909,0.597800
22,A30,1104,0.596920
129,C65,1259,0.596108
116,C43,950,0.594211
11,A11,1076,0.592937


In [7]:
top_white = opening_stats.head(15)

fig = px.bar(
    top_white,
    x="eco",
    y="white_score",
    text="white_score",
    title="Best Openings for White (min 500 games)"
)

fig.update_layout(
    xaxis_title="ECO Code",
    yaxis_title="White Score"
)

fig.show()

In [8]:
draw_stats = []

for eco, group in games.groupby("eco"):

    games_count = len(group)

    if games_count < 500:
        continue

    draw_rate = (
        (
            group["result"]
            == "1/2-1/2"
        ).mean()
        * 100
    )

    draw_stats.append([
        eco,
        games_count,
        draw_rate
    ])

draw_df = pd.DataFrame(
    draw_stats,
    columns=[
        "eco",
        "games",
        "draw_rate"
    ]
)

draw_df = draw_df.sort_values(
    "draw_rate",
    ascending=False
)

draw_df.head(20)

,eco,games,draw_rate
130,C67,1230,54.878049
158,D41,580,40.689655
137,C84,1048,34.446565
120,C47,1991,34.304370
116,C43,950,32.526316
151,D27,779,32.349166
115,C42,3084,32.198444
157,D38,2084,31.094050
179,E49,686,31.049563
138,C88,1129,29.937998


In [9]:
fig = px.bar(
    draw_df.head(15),
    x="eco",
    y="draw_rate",
    text="draw_rate",
    title="Most Drawish Openings (min 500 games)"
)

fig.update_layout(
    xaxis_title="ECO Code",
    yaxis_title="Draw Rate (%)"
)

fig.show()

In [10]:
carlsen = games[
    games["white"]
    .str.contains(
        "Carlsen",
        case=False,
        na=False
    )
]

print(
    f"Games found: {len(carlsen)}"
)

carlsen["eco"]\
.value_counts()\
.head(15)

Games found: 464


eco
A00    32
A45    20
A07    14
B50    13
D37    12
B10    12
A01    12
A14    10
B20    10
D00     9
C50     9
A20     8
D31     8
D35     8
C00     8
Name: count, dtype: int64

In [11]:
caruana = games[
    games["white"]
    .str.contains(
        "Caruana",
        case=False,
        na=False
    )
]

print(
    f"Games found: {len(caruana)}"
)

caruana["eco"]\
.value_counts()\
.head(15)

Games found: 616


eco
C20    49
D35    24
B40    20
C00    15
C45    14
C50    13
A00    12
A45    12
B23    12
B00    12
D31    12
A15    11
B20    11
A13    10
A20    10
Name: count, dtype: int64

In [12]:
erigaisi = games[
    games["white"]
    .str.contains(
        "Erigaisi",
        case=False,
        na=False
    )
]

print(
    f"Games found: {len(erigaisi)}"
)

erigaisi["eco"]\
.value_counts()\
.head(15)

Games found: 555


eco
E10    31
A15    27
A13    23
D30    22
B50    20
A09    20
A45    18
A04    17
A01    16
C53    14
D37    13
C00    12
A00    10
E73    10
C42     9
Name: count, dtype: int64

In [13]:
firouzja = games[
    games["white"]
    .str.contains(
        "Firouzja",
        case=False,
        na=False
    )
]

print(
    f"Games found: {len(firouzja)}"
)

firouzja["eco"]\
.value_counts()\
.head(15)

Games found: 360


eco
C53    16
B90    13
D02    13
D20    11
E11    11
A45     9
D37     9
C50     8
A05     8
A07     8
B30     8
C65     8
A06     8
D00     7
C11     7
Name: count, dtype: int64

In [14]:
players = [
    "Carlsen",
    "Caruana",
    "Firouzja",
    "Erigaisi"
]

comparison = []

for player in players:

    player_games = games[
        games["white"]
        .str.contains(
            player,
            case=False,
            na=False
        )
    ]

    top_eco = (
        player_games["eco"]
        .value_counts()
        .head(1)
    )

    comparison.append([
        player,
        len(player_games),
        top_eco.index[0],
        top_eco.iloc[0]
    ])

comparison = pd.DataFrame(
    comparison,
    columns=[
        "player",
        "games",
        "favorite_eco",
        "times_played"
    ]
)

comparison


,player,games,favorite_eco,times_played
0,Carlsen,464,A00,32
1,Caruana,616,C20,49
2,Firouzja,360,C53,16
3,Erigaisi,555,E10,31


In [15]:
carlsen_stats = []

for eco, group in carlsen.groupby("eco"):

    if len(group) < 5:
        continue

    score = (
        (
            group["result"] == "1-0"
        ).sum()
        +
        0.5 *
        (
            group["result"] == "1/2-1/2"
        ).sum()
    ) / len(group)

    carlsen_stats.append(
        [
            eco,
            len(group),
            score
        ]
    )

carlsen_stats = pd.DataFrame(
    carlsen_stats,
    columns=[
        "eco",
        "games",
        "score"
    ]
)

carlsen_stats.sort_values(
    "score",
    ascending=False
).head(15)

,eco,games,score
21,D35,8,0.937500
20,D31,8,0.937500
1,A01,12,0.916667
0,A00,32,0.906250
3,A07,14,0.892857
16,C26,7,0.857143
6,A45,20,0.850000
2,A06,6,0.833333
13,B40,7,0.785714
14,B50,13,0.769231
